# 05 — GAN Quality Evaluation

Run all three quality checks before proceeding to the pricing network:
1. **Distribution matching** — histogram of log returns, kurtosis comparison
2. **Volatility clustering** — autocorrelation of squared returns
3. **MMD score** — baseline vs GAN maximum mean discrepancy

**Do not proceed to training data generation until these checks pass.**

In [ ]:
import sys, os
import numpy as np
sys.path.insert(0, os.path.abspath('..'))

from src.evaluation.gan_eval import (
    plot_return_distribution,
    plot_volatility_clustering,
    run_mmd_check,
    run_all_quality_checks,
)

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
FIGURES_DIR = os.path.join('..', 'results', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
# Load real training sequences and synthetic sequences
real_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))
fake_seqs = np.load(os.path.join(PROCESSED_DIR, 'gan_sequences.npy'))

print(f'Real training sequences: {real_seqs.shape}')
print(f'Synthetic sequences:     {fake_seqs.shape}')

In [ ]:
# Split real data into train/val for MMD baseline
n_val = len(real_seqs) // 5
real_val = real_seqs[-n_val:]
real_train = real_seqs[:-n_val]
print(f'Real train for MMD: {real_train.shape}')
print(f'Real val for MMD:   {real_val.shape}')

## Run All Quality Checks

In [ ]:
results = run_all_quality_checks(real_train, real_val, fake_seqs, figures_dir=FIGURES_DIR)

## Summary

In [ ]:
print('=' * 50)
print('GAN Quality Check Summary')
print('=' * 50)
print(f'Real kurtosis:     {results["real_kurtosis"]:.2f}  (SPY ~4-5, Gaussian = 3)')
print(f'Synthetic kurtosis: {results["fake_kurtosis"]:.2f}')
print(f'Baseline MMD:      {results["baseline_mmd"]:.6f}')
print(f'GAN MMD:           {results["gan_mmd"]:.6f}')
print(f'Ratio:             {results["ratio"]:.2f}x')
print(f'MMD check:         {"PASS" if results["passed"] else "FAIL"}')
print()
if results['passed']:
    print('All checks passed. Proceed to training data generation.')
else:
    print('MMD check FAILED. Consider retraining the GAN with more epochs or tuning hyperparameters.')